# Clase 4: Ingeniería de Prompt

Un modelo de lenguaje grande (LLM), es una red neuronal profunda basada en la arquitectura Transformer, entrenada sobre grandes volúmenes de texto mediante aprendizaje auto-supervisado. Su tarea fundamental es modelar la probabilidad del siguiente token en una secuencia: dado un contexto de entrada, el modelo estima cuál es el token más probable que viene a continuación.
Para lograrlo, el texto se divide en tokens —unidades de texto como palabras, subpalabras o caracteres— y cada token se convierte en un vector numérico llamado embedding. Estos vectores forman una matriz que atraviesa el Transformer, donde cálculos sucesivos transforman la información incorporando el contexto completo de la secuencia. Al final, el modelo produce logits: puntuaciones sobre todo el vocabulario. El token con la puntuación más alta es el predicho como siguiente.
Repitiendo esta operación token por token, el modelo genera texto coherente y contextualmente relevante.

![Diagrama Pipeline LLM](img/llm_pipeline.svg)

# 1. Criterios Explícitos y Few-Shot Prompting

<h2 style="color: #e67e22;">Por qué los criterios vagos fallan</h2>

Una instrucción como “check that comments are accurate” es ambigua. Claude interpreta “accurate” de forma amplia y termina flaggeando comentarios que simplemente no están actualizados, incluso si no causan confusión real.

La alternativa efectiva es un criterio categórico: “flag comments only when the claimed behavior directly contradicts what the code does”. Esto reduce drásticamente los falsos positivos porque Claude tiene un umbral claro para decidir qué reportar.

![Diagrama Pipeline LLM](img/matriz_confusion_2.svg)

Donde:

- **TP** = verdaderos positivos (flag correcto)
- **FP** = falsos positivos (flaggea algo que estaba bien)
- **FN** = falsos negativos (no flaggea un error real)
- **TN** = verdaderos negativos (silencio correcto)

#### Precisión

```
Precisión = TP / (TP + FP)
```

De lo que flaggeé, ¿cuánto era real? Sensible a los FP.

#### Recall (exhaustividad)

```
Recall = TP / (TP + FN)
```

De lo real, ¿cuánto capturé? Sensible a los FN.

#### Exactitud (accuracy)

```
Accuracy = (TP + TN) / (TP + TN + FP + FN)
```


#### Estrategia de restauración de confianza
1. Identificar categorías con alta tasa de falsos positivos
2. Deshabilitar temporalmente esas categorías
3. Refinar los criterios con ejemplos específicos
4. Re-habilitar gradualmente con criterios categóricos
5. Monitorear la tasa de dismissal por categoría

<span style="color: red;">**Para el examen:** </span>
Si un escenario describe que developers ignoran findings de un review tool, la respuesta correcta involucra refinar criterios específicos o deshabilitar categorías problemáticas — no “mejorar el modelo” ni “agregar más contexto general”.




---

<h2 style="color: #e67e22;">Few-Shot Prompting</h2>

Few-shot es la técnica más efectiva cuando instrucciones detalladas producen output inconsistente. En lugar de describir qué hacer, **muestras qué hacer con ejemplos concretos.**

#### Cuándo usar few-shot
- Instrucciones detalladas producen output variable entre invocaciones
- La tarea requiere juicio contextual (no solo pattern matching)
- Necesitas que Claude generalice a casos nuevos similares
- Hay ambigüedad inherente en la selección de herramientas o categorías

#### Estructura de un few-shot example
Cada ejemplo debe incluir:

**1. Input:** El caso concreto.  
**2. Output esperado:** La acción o decisión correcta.  
**3. Reasoning:** Por qué se eligió esa acción (esto es clave para generalización).  

#### Few-shot para tool selection
Cuando Claude debe elegir entre múltiples herramientas disponibles, few-shot examples resuelven la ambigüedad mejor que descripciones largas de cada tool.

#### Cantidad óptima: 2-4 examples
- **1 example:** Insuficiente para establecer un patrón
- **2-4 examples:** Punto óptimo entre cobertura y eficiencia de tokens
- **5+ examples:** Rendimientos decrecientes, consume contexto valioso

<span style="color: red;">Para el examen:</span>
Few-shot prompting es la respuesta correcta cuando el escenario describe output inconsistente a pesar de instrucciones detalladas, o cuando se necesita manejar ambigüedad en tool selection.

#### Combinando criterios explícitos con few-shot
La combinación más poderosa es:

Criterios categóricos para definir el scope (qué reportar/ignorar)
Few-shot examples para calibrar el juicio dentro del scope
Reasoning en los examples para que Claude generalice correctamente
Esta combinación reduce falsos positivos manteniendo alta cobertura de issues reales.




# 2. Structured Output con tool_use y JSON Schemas

<h2 style="color: #e67e22;">tool_use como mecanismo de structured output</h2>

El approach más confiable para obtener output estructurado de Claude es tool_use con JSON schemas. En lugar de pedir “responde en JSON”, defines una herramienta cuyo input schema es exactamente la estructura que necesitas, y Claude la “invoca” con datos conformes al schema.

Esto no es un hack — es el mecanismo diseñado por Anthropic para structured output garantizado.



---

<h2 style="color: #e67e22;">Diseño de JSON Schemas efectivos</h2>

**required vs optional fields**  
No todos los campos deben ser required. Si el source document puede no contener cierta información, hacer el campo required fuerza a Claude a inventar un valor.

**Patrón correcto:** Campos que siempre existen en el source como required, campos que pueden faltar como nullable.

---

<h2 style="color: #e67e22;">Format normalization en prompts</h2>

Strict schemas garantizan estructura pero no formato. Si necesitas que las fechas sean YYYY-MM-DD y los montos tengan 2 decimales, debes especificarlo en el prompt:

**Format rules:**
- Dates: YYYY-MM-DD format
- Amounts: numeric with exactly 2 decimal places
- Phone numbers: E.164 format (+1234567890)
- Names: Title Case  

Estas reglas van en el **prompt**, no en el schema (JSON Schema tiene format pero Claude no siempre lo respeta para normalización).

---

<h2 style="color: #e67e22;">Errores semánticos: lo que schemas NO resuelven</h2>

**Strict schemas eliminan:**. 
- JSON mal formado
- Campos faltantes
- Tipos incorrectos (string donde debería ser number)

**Pero NO eliminan:**

- Valores extraídos incorrectamente
- Datos inventados (alucinaciones)
- Sumas que no cuadran (total ≠ sum of line items)
- Datos colocados en el campo equivocado
- Para errores semánticos, necesitas validación post-extracción (siguiente capítulo).

<h2 style="color: #e67e22;">Ejemplo en Python con Pydantic</h2>

In [9]:
import json
from datetime import date

import anthropic
from pydantic import (
    BaseModel, Field, EmailStr
    )

In [10]:
client = anthropic.Anthropic()
MODELO = "claude-sonnet-4-6"

In [11]:
texto = """
Confirmación de pedido #A-1029.
Fecha: 03/02/2025.
Cliente: María González (maria.gonzalez@empresa.com).
Importe total: 1.250,50 €.
"""
print(texto)


Confirmación de pedido #A-1029.
Fecha: 03/02/2025.
Cliente: María González (maria.gonzalez@empresa.com).
Importe total: 1.250,50 €.



In [14]:
class Pedido(BaseModel):
    cliente: str = Field(
        description="Nombre completo del cliente."
    )
    email: EmailStr = Field(
        description="Correo electrónico de contacto del cliente."
    )
    fecha: date = Field(
        description="Fecha del pedido. En el texto está en formato español DD/MM/AAAA."
    )
    importe: float = Field(
        description="Importe total en euros, como número decimal con punto."
    )
    telefono: str | None = Field(
        default=None,
        description="Teléfono de contacto en formato E.164 (+34...). "
                    "Si no aparece en el texto, déjalo en null; no lo inventes.",
    )

print(json.dumps(Pedido.model_json_schema(), indent=2, ensure_ascii=False))

{
  "properties": {
    "cliente": {
      "description": "Nombre completo del cliente.",
      "title": "Cliente",
      "type": "string"
    },
    "email": {
      "description": "Correo electrónico de contacto del cliente.",
      "format": "email",
      "title": "Email",
      "type": "string"
    },
    "fecha": {
      "description": "Fecha del pedido. En el texto está en formato español DD/MM/AAAA.",
      "format": "date",
      "title": "Fecha",
      "type": "string"
    },
    "importe": {
      "description": "Importe total en euros, como número decimal con punto.",
      "title": "Importe",
      "type": "number"
    },
    "telefono": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Teléfono de contacto en formato E.164 (+34...). Si no aparece en el texto, déjalo en null; no lo inventes.",
      "title": "Telefono"
    }
  },
  "required": [
    "cliente"

In [15]:
def extraer(modelo, instruccion=""):
    """Extrae datos del texto. `instruccion` añade contexto extra al prompt."""
    prompt = "Extrae los datos del siguiente texto."
    if instruccion:
        prompt += " " + instruccion
    prompt += f"\n\n{texto}"

    r = client.messages.parse(
        model=MODELO,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
        output_format=modelo,
    )
    return r.parsed_output

In [16]:
p1 = extraer(Pedido)
print(p1.model_dump_json(indent=2))

{
  "cliente": "María González",
  "email": "maria.gonzalez@empresa.com",
  "fecha": "2025-02-03",
  "importe": 1250.5,
  "telefono": null
}


### Ejemplo del DTO en TypeScript
```javascript
import { z } from "zod";
import { zodOutputFormat } from "@anthropic-ai/sdk/helpers/zod";

const PedidoSchema = z.object({
  cliente: z.string().describe("Nombre completo del cliente."),
  email: z.string().email().describe("Correo electrónico de contacto del cliente."),
  fecha: z.iso
    .date()
    .describe("Fecha del pedido. En el texto está en formato español DD/MM/AAAA."),
  importe: z.number().describe("Importe total en euros, como número decimal con punto."),
  telefono: z
    .string()
    .nullable()
    .default(null)
    .describe(
      "Teléfono de contacto en formato E.164 (+34...). " +
        "Si no aparece en el texto, déjalo en null; no lo inventes.",
    ),
});

// equivalente a: print(json.dumps(Pedido.model_json_schema(), ...))
console.dir(zodOutputFormat(PedidoSchema), { depth: null });
```

# 3. Validación, Retry y Feedback Loops
El uso de `tool_use` con JSON schemas elimina errores de sintaxis, pero no errores semánticos. Para producción se necesita una capa de validación, retry con feedback específico y bucles de mejora continua.

<h2 style="color: #e67e22;">Errores semánticos: lo que el schema no atrapa</h2>

Un JSON schema garantiza forma pero no significado. Ejemplos de errores que pasan la validación de schema pero son semánticamente inválidos:

- Una factura con line_items cuyos subtotales no suman al total declarado.
- Un campo shipping_address que en realidad contiene la billing_address (ambos son strings válidos).
- Un campo publication_date con valor "2024-01-01" cuando el documento es de 2026.
Detectar esto requiere validación a nivel de aplicación, no solo de schema.

---

<h2 style="color: #e67e22;">Retry-with-error-feedback</h2>

El patrón clave: cuando la validación falla, no se reintenta a ciegas. Se le manda al modelo el error específico y la extracción fallida.

#### Cuándo el retry NO va a funcionar

El retry es efectivo para errores de formato y estructura, pero inútil cuando la información requerida simplemente no está en el documento fuente.

#### Anti-patterns
1. **Reintentos a ciegas**
Repetir el mismo prompt sin agregar el error específico tiene baja probabilidad de éxito. El modelo no sabe qué cambiar.

2. **Reintentos infinitos**
Si la información no está en el documento, mil reintentos no la van a inventar. Necesitás un cap razonable (ej: 2 retries) y un fallback.

3. **Suprimir errores de validación**
Marcar como exitosa una extracción que falló validación contamina los datos downstream. Mejor fallar con un mensaje claro o rutear a revisión humana.



---

<h2 style="color: #e67e22;">Validaciones cruzadas auto-correctivas</h2>

- Pedirle al modelo que extraiga dos versiones de un mismo dato y que marque discrepancias.

- Tener un modelo reviewer aparte.